# Band-edge loop-gain retuning at 1.24 R_s

A companion notebook for `notes/band-edge-loop-gain-retuning.md`.

This keeps the adjacent-channel setup fixed at the first half-sine settle-point spacing and moves only one knob: loop gain.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPTS = REPO / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from waveform_carrier_front_ends import band_edge_closed_loop_row, study_band_edge_closed_loop_gain_sweep


In [ ]:
gains = [0.0005, 0.0010, 0.0015, 0.0020, 0.0030, 0.0040, 0.0050, 0.0060, 0.0080, 0.0100, 0.0120, 0.0150, 0.0180, 0.0200, 0.0220, 0.0240]
rows = study_band_edge_closed_loop_gain_sweep(gains, adjacent_relative_power_db=0.0, channel_spacing=1.24)
rows[:4]


In [ ]:
lookup = {(row.design, round(row.loop_gain, 4)): row for row in rows}
summary = []
for gain in [0.0020, 0.0200, 0.0220]:
    proxy = lookup[('proxy_bandpass', round(gain, 4))]
    half = lookup[('gnuradio_half_sine', round(gain, 4))]
    summary.append({
        'gain': round(gain, 4),
        'proxy_mean_tail_residual': round(proxy.tail_mean_abs_residual_cfo, 6),
        'proxy_tail_fraction': round(proxy.tail_within_threshold_fraction, 3),
        'half_sine_mean_tail_residual': round(half.tail_mean_abs_residual_cfo, 6),
        'half_sine_tail_fraction': round(half.tail_within_threshold_fraction, 3),
        'residual_ratio': round(half.tail_mean_abs_residual_cfo / proxy.tail_mean_abs_residual_cfo, 3),
    })
summary


In [ ]:
seed_pairs = [(19, 173), (23, 211), (31, 271), (47, 389)]
sensitivity = []
for gain in [0.0190, 0.0200, 0.0210, 0.0220]:
    fractions = []
    for desired_seed, adjacent_seed in seed_pairs:
        row = band_edge_closed_loop_row(
            'gnuradio_half_sine',
            adjacent_enabled=True,
            adjacent_relative_power_db=0.0,
            channel_spacing=1.24,
            loop_gain=gain,
            desired_seed=desired_seed,
            adjacent_seed=adjacent_seed,
        )
        fractions.append(row.tail_within_threshold_fraction)
    sensitivity.append({'gain': gain, 'fractions': fractions})
sensitivity


## Readout

The useful conclusion is not just that smaller gain helps. It is that the **ordering stays the same** while gain moves.

- Lower gain reduces adjacent pull for both lanes.
- The half-sine residual stays far above the proxy residual across the tested gain set.
- The half-sine lane also starts losing settle margin first as gain rises past about `0.020–0.022`.

That is enough to stop treating the spacing result as an unfinished tuning artifact.